# Launch a GINE Training on Colab

This notebook launches `configs/gine/shared_gine_a1_no_entropy_decay_s0_det.toml` without `sbatch`.

Use it in Colab/VSCode after cloning or opening the `RL_Marl2grid` repo. Enable a GPU runtime before running.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def run(cmd, check=True, **kwargs):
    print("$", " ".join(map(str, cmd)))
    if shutil.which(str(cmd[0])) is None:
        print(f"Command not found: {cmd[0]}")
        return None
    return subprocess.run(cmd, check=check, **kwargs)

print("Python:", sys.version)
if shutil.which('nvidia-smi'):
    run(['nvidia-smi'], check=False)
else:
    print('nvidia-smi not found. If you are on Colab, enable Runtime -> Change runtime type -> GPU, then reconnect.')

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
nvidia-smi not found. If you are on Colab, enable Runtime -> Change runtime type -> GPU, then reconnect.


## Locate the Repo

If this fails, set `REPO_ROOT` manually to the folder that contains `Topology_Task`.

In [3]:
!git clone https://github.com/corentinplumet/RL_Marl2grid.git

Cloning into 'RL_Marl2grid'...
remote: Enumerating objects: 897, done.
remote: Counting objects: 100% (897/897), done.
remote: Compressing objects: 100% (465/465), done.
remote: Total 897 (delta 514), reused 794 (delta 413), pack-reused 0 (from 0)
Receiving objects: 100% (897/897), 28.03 MiB | 20.68 MiB/s, done.
Resolving deltas: 100% (514/514), done.


In [4]:
cd RL_Marl2grid

/content/RL_Marl2grid


In [5]:
!git fetch
!git switch gnn-grid-encoders

Already on 'gnn-grid-encoders'
Your branch is up to date with 'origin/gnn-grid-encoders'.


In [6]:
def find_repo_root():
    starts = [Path.cwd(), Path.home(), Path('/content/RL_Marl2grid')]
    for start in starts:
        if not start.exists():
            continue
        for path in [start, *start.parents]:
            if (path / 'Topology_Task' / 'run_from_config.py').is_file():
                return path
    raise FileNotFoundError(
        'Could not find RL_Marl2grid. Clone/open the repo, or set REPO_ROOT manually.'
    )

REPO_ROOT = find_repo_root()
TASK_DIR = REPO_ROOT / 'Topology_Task'
os.chdir(TASK_DIR)
print('REPO_ROOT =', REPO_ROOT)
print('TASK_DIR =', TASK_DIR)
print('Config exists:', Path('configs/gine/shared_gine_a1_no_entropy_decay_s0_det.toml').is_file())

REPO_ROOT = /content/RL_Marl2grid
TASK_DIR = /content/RL_Marl2grid/Topology_Task
Config exists: True


## Install Dependencies

Run this once per fresh Colab runtime. This installs Grid2Op from the `dev_multiagent` branch because the training code imports `grid2op.multi_agent`.

In [7]:
def pip_install(*packages, extra_args=None, check=True):
    cmd = [sys.executable, '-m', 'pip', 'install', '--prefer-binary', *packages]
    if extra_args:
        cmd.extend(extra_args)
    return run(cmd, check=check)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])

scipy_package = 'scipy==1.11.3' if sys.version_info < (3, 12) else 'scipy>=1.14,<2'
print('Using', scipy_package, 'for Python', sys.version.split()[0])

base_packages = [
    'numpy<2',
    'tomli',
    'dm-tree==0.1.10',
    'lz4',
    'pandas',
    'matplotlib',
    'packaging',
    'networkx',
    'cloudpickle',
    'tqdm',
    'PyYAML',
    'psutil',
    'protobuf>=4.25,<6',
    'wandb==0.25.0',
    'gymnasium==0.29.1',
    'stable-baselines3==2.3.2',
    scipy_package,
    'pandapower==2.14.11',
    'deepdiff',
    'pybind11',
    'ray[rllib]==2.55.1',
    'lightsim2grid==0.9',
]

for package in base_packages:
    print('\nInstalling', package)
    pip_install(package)

grid2op_repo = Path('/content/grid2op-dev-multiagent') if Path('/content').exists() else Path.home() / 'grid2op-dev-multiagent'
run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'grid2op', 'Grid2Op'], check=False)
if grid2op_repo.exists():
    run(['git', '-C', str(grid2op_repo), 'fetch', 'origin', 'dev_multiagent'])
    run(['git', '-C', str(grid2op_repo), 'checkout', 'dev_multiagent'])
    run(['git', '-C', str(grid2op_repo), 'pull', '--ff-only'])
else:
    run(['git', 'clone', '--branch', 'dev_multiagent', 'https://github.com/Grid2op/grid2op.git', str(grid2op_repo)])

cluster_utils = grid2op_repo / 'grid2op' / 'cluster_utils'
cluster_utils.mkdir(parents=True, exist_ok=True)
(cluster_utils / '__init__.py').touch()
pip_install('--no-deps', '-e', str(grid2op_repo))

import torch
print('Torch:', torch.__version__)

pip_install('torch_geometric')

# Optional PyG compiled extensions can speed up some operations, but Colab's
# default Torch/CUDA build often has no matching wheels. The GINE encoder can
# run with torch_geometric alone, so leave this off unless you really need it.
INSTALL_PYG_EXTENSIONS = False
if INSTALL_PYG_EXTENSIONS:
    torch_version = torch.__version__.split('+')[0]
    cuda_version = torch.version.cuda
    cuda_tag = 'cpu' if cuda_version is None else 'cu' + cuda_version.replace('.', '')
    pyg_wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
    print('PyG wheel index:', pyg_wheel_url)
    pip_install(
        'pyg_lib',
        'torch_scatter',
        'torch_sparse',
        'torch_cluster',
        'torch_spline_conv',
        extra_args=['-f', pyg_wheel_url],
        check=False,
    )

import grid2op
import lightsim2grid
import ray.rllib
import torch_geometric
from torch import nn
from grid2op.multi_agent import MultiAgentEnv
from torch_geometric.nn import GINEConv
x = torch.randn(3, 8)
edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
edge_attr = torch.randn(4, 8)
conv = GINEConv(nn.Sequential(nn.Linear(8, 8), nn.ReLU(), nn.Linear(8, 8)), edge_dim=8)
_ = conv(x, edge_index, edge_attr=edge_attr)
print('grid2op:', grid2op.__version__)
print('lightsim2grid:', lightsim2grid.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('Grid2Op multi_agent import: ok')
print('GINEConv smoke test: ok')

$ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
Using scipy>=1.14,<2 for Python 3.12.13

Installing numpy<2
$ /usr/bin/python3 -m pip install --prefer-binary numpy<2

Installing tomli
$ /usr/bin/python3 -m pip install --prefer-binary tomli

Installing dm-tree==0.1.10
$ /usr/bin/python3 -m pip install --prefer-binary dm-tree==0.1.10

Installing lz4
$ /usr/bin/python3 -m pip install --prefer-binary lz4

Installing pandas
$ /usr/bin/python3 -m pip install --prefer-binary pandas

Installing matplotlib
$ /usr/bin/python3 -m pip install --prefer-binary matplotlib

Installing packaging
$ /usr/bin/python3 -m pip install --prefer-binary packaging

Installing networkx
$ /usr/bin/python3 -m pip install --prefer-binary networkx

Installing cloudpickle
$ /usr/bin/python3 -m pip install --prefer-binary cloudpickle

Installing tqdm
$ /usr/bin/python3 -m pip install --prefer-binary tqdm

Installing PyYAML
$ /usr/bin/python3 -m pip install --prefer-binary PyYAML

Installing psutil
$ /u

/content/grid2op-dev-multiagent/grid2op/Action/baseAction.py:179: SyntaxWarning: invalid escape sequence '\['
  by the action (in this case :attr:`BaseAction._subs_impacted`\[sub_id\] is ``True``) or not
/content/grid2op-dev-multiagent/grid2op/Action/baseAction.py:1272: SyntaxWarning: invalid escape sequence '\*'
  n_gen, n_load, n_line, sub_info, dim_topo, all vectors \*_to_subid, and \*_pos_topo_vect are
/content/grid2op-dev-multiagent/grid2op/Action/baseAction.py:4968: SyntaxWarning: invalid escape sequence '\*'
  * set to bus 1 the (unique) element for which \*_pos_topo_vect is 1
/content/grid2op-dev-multiagent/grid2op/Action/baseAction.py:5287: SyntaxWarning: invalid escape sequence '\*'
  * change the bus of the (unique) element for which \*_pos_topo_vect is 1
/content/grid2op-dev-multiagent/grid2op/Action/baseAction.py:6823: SyntaxWarning: invalid escape sequence '\s'
  $\sum_{\text{all generators } g} p^{(g, scenario)}_t = \sum_{\text{controlable generators } c}  p^{(c)}_t + \s

grid2op: 1.11.0.dev4
lightsim2grid: 0.9.0
torch_geometric: 2.7.0
Grid2Op multi_agent import: ok
GINEConv smoke test: ok


## Optional W&B Login

For the short speed test below, W&B is disabled. For a full online run, set `USE_WANDB = True` and run this cell.

In [8]:
USE_WANDB = False

if USE_WANDB:
    try:
        import wandb
        wandb.login()
    except ImportError as exc:
        print('W&B import failed. For the speed test, keep USE_WANDB = False.')
        print('To repair W&B manually, try: python -m pip install --force-reinstall --no-cache-dir "protobuf>=4.25,<6" "wandb==0.25.0"')
        raise
else:
    print('Skipping W&B login.')

Skipping W&B login.


## Launch Training

`SHORT_SPEED_TEST = True` keeps the run to two PPO rollouts for timing. Set it to `False` to run the full config.

In [9]:
CONFIG = 'configs/gine/shared_gine_a1_no_entropy_decay_s0_det.toml'
SHORT_SPEED_TEST = True
LOG_PATH = Path('colab_gine_training.log')

def run_streaming(cmd, log_path):
    print('$', ' '.join(map(str, cmd)))
    with log_path.open('w') as log_file:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
        return_code = proc.wait()

    if return_code != 0:
        lines = log_path.read_text(errors='replace').splitlines()
        print('\n===== Last 120 log lines =====')
        print('\n'.join(lines[-120:]))
        raise RuntimeError(f'Training failed with exit code {return_code}. Full log: {log_path.resolve()}')

    print(f'Finished successfully. Full log: {log_path.resolve()}')

cmd = [
    sys.executable,
    '-u',
    'run_from_config.py',
    CONFIG,
    '--cuda', 'true',
]

if SHORT_SPEED_TEST:
    cmd += [
        '--total-timesteps', '80000',
        '--eval-episodes', '1',
        '--track', 'false',
        '--wandb-mode', 'disabled',
    ]

run_streaming(cmd, LOG_PATH)

$ /usr/bin/python3 -u run_from_config.py configs/gine/shared_gine_a1_no_entropy_decay_s0_det.toml --cuda true --total-timesteps 80000 --eval-episodes 1 --track false --wandb-mode disabled
========== JED config run ==========
Config: /content/RL_Marl2grid/Topology_Task/configs/gine/shared_gine_a1_no_entropy_decay_s0_det.toml
Run name: shared_gine_a1_no_entropy_decay_s0_det
Run dir: /content/RL_Marl2grid/outputs/jed-shared_gine_a1_no_entropy_decay_s0_det-local
Task dir: /content/RL_Marl2grid/Topology_Task
Job id: local
Array task id: none
CPUs per task: 72
Command: /usr/bin/python3 -u main.py --time-limit 1300 --checkpoint true --alg MAPPO --seed 0 --verbose true --exp-tag shared_gine_a1_no_entropy_decay_s0_det --track true --wandb-project Grid2Op --wandb-entity corentin-plumet-epfl --wandb-mode online --th-deterministic false --cuda false --n-threads 20 --env-id bus14 --n-envs 20 --action-type topology --difficulty 0 --decentralized true --n1-reward false --env-config-path scenario.json

: 

: 

: 